In [1]:
import subprocess
subprocess.run(['pip', 'install', 'plotly', 'nbformat', '--upgrade'])

CompletedProcess(args=['pip', 'install', 'plotly', 'nbformat', '--upgrade'], returncode=0)

In [2]:
import plotly.io as pio
import plotly.express as px
import plotly.graph_objects as go
import pandas as pd

# Set renderer for Jupyter
pio.renderers.default = 'notebook'

# Load data
df = pd.read_csv('../data/west_africa_solar_data_2014_2025.csv')

# Prepare datasets
monthly_avg = df.groupby(['City', 'Country', 'Climate_Zone',
                           'Month'])['GHI'].mean().reset_index()

annual_avg = df.groupby(['City', 'Country', 'Climate_Zone',
                          'Year'])['GHI'].mean().reset_index()

city_summary = df.groupby(['City', 'Country',
                            'Climate_Zone'])['GHI'].mean().reset_index()
city_summary.columns = ['City', 'Country', 'Climate_Zone', 'Mean_GHI']
city_summary = city_summary.sort_values('Mean_GHI', ascending=False)

print(f"✓ Data loaded: {len(df)} records")
print(f"✓ Plotly renderer: {pio.renderers.default}")

✓ Data loaded: 2574 records
✓ Plotly renderer: notebook


In [3]:
import plotly.io as pio
import plotly.express as px
import plotly.graph_objects as go
import pandas as pd
import webbrowser
import os

# Set renderer
pio.renderers.default = 'browser'

# Helper function to save and open charts
def show_chart(fig, filename):
    filepath = f'../reports/{filename}'
    fig.write_html(filepath)
    abs_path = os.path.abspath(filepath)
    webbrowser.open(f'file:///{abs_path}')
    print(f"✓ Chart saved and opened: {filename}")

# Load data
df = pd.read_csv('../data/west_africa_solar_data_2014_2025.csv')

monthly_avg = df.groupby(['City', 'Country', 'Climate_Zone',
                           'Month'])['GHI'].mean().reset_index()
annual_avg = df.groupby(['City', 'Country', 'Climate_Zone',
                          'Year'])['GHI'].mean().reset_index()
city_summary = df.groupby(['City', 'Country',
                            'Climate_Zone'])['GHI'].mean().reset_index()
city_summary.columns = ['City', 'Country', 'Climate_Zone', 'Mean_GHI']
city_summary = city_summary.sort_values('Mean_GHI', ascending=False)

print(f"✓ Data loaded: {len(df)} records")

✓ Data loaded: 2574 records


In [4]:
# ================================================
# CHART 1 — Interactive Bar Chart
# ================================================

zone_colors = {
    'Coastal Humid': '#0099ff',
    'Savanna':       '#00c27c',
    'Sahel':         '#f7c948',
    'Desert/Arid':   '#ff6b35'
}

fig = px.bar(
    city_summary,
    x='Mean_GHI',
    y='City',
    color='Climate_Zone',
    color_discrete_map=zone_colors,
    orientation='h',
    title='Annual Average Solar Irradiance — 18 West African Cities (2014–2025)',
    labels={
        'Mean_GHI': 'Annual Average GHI (kWh/m²/day)',
        'City': '',
        'Climate_Zone': 'Climate Zone'
    },
    text='Mean_GHI'
)

fig.update_traces(texttemplate='%{text:.2f}', textposition='outside')

fig.update_layout(
    height=600,
    plot_bgcolor='white',
    paper_bgcolor='white',
    font=dict(family='Arial', size=12),
    title_font_size=14,
    xaxis=dict(showgrid=True, gridcolor='#eeeeee'),
    yaxis=dict(categoryorder='total ascending')
)

fig.write_html('../reports/plotly_city_ranking.html')
show_chart(fig, 'plotly_city_ranking.html')

✓ Chart saved and opened: plotly_city_ranking.html


In [5]:
# ================================================
# CHART 2 — Interactive Line Chart
# ================================================

fig2 = px.line(
    annual_avg,
    x='Year',
    y='GHI',
    color='City',
    title='Annual Average GHI Trends — 18 West African Cities (2014–2025)',
    labels={
        'GHI': 'Annual Average GHI (kWh/m²/day)',
        'Year': 'Year',
        'City': 'City'
    },
    markers=True
)

fig2.update_layout(
    height=550,
    plot_bgcolor='white',
    paper_bgcolor='white',
    font=dict(family='Arial', size=12),
    title_font_size=14,
    xaxis=dict(showgrid=True, gridcolor='#eeeeee', dtick=1),
    yaxis=dict(showgrid=True, gridcolor='#eeeeee'),
    hovermode='x unified'
)

show_chart(fig2, 'plotly_ghi_trends.html')

✓ Chart saved and opened: plotly_ghi_trends.html


In [6]:
# ================================================
# CHART 3 — Animated Chart
# Solar irradiance by month across years
# ================================================

month_names = {1:'Jan', 2:'Feb', 3:'Mar', 4:'Apr', 5:'May', 6:'Jun',
               7:'Jul', 8:'Aug', 9:'Sep', 10:'Oct', 11:'Nov', 12:'Dec'}

# Add month names
df_anim = df.copy()
df_anim['Month_Name'] = df_anim['Month'].map(month_names)
df_anim['Month_Name'] = pd.Categorical(df_anim['Month_Name'],
                                        categories=list(month_names.values()),
                                        ordered=True)

# Average GHI by city, year, month
anim_data = df_anim.groupby(['City', 'Country', 'Climate_Zone',
                               'Year', 'Month_Name'])['GHI'].mean().reset_index()

zone_colors = {
    'Coastal Humid': '#0099ff',
    'Savanna':       '#00c27c',
    'Sahel':         '#f7c948',
    'Desert/Arid':   '#ff6b35'
}

fig3 = px.bar(
    anim_data,
    x='City',
    y='GHI',
    color='Climate_Zone',
    color_discrete_map=zone_colors,
    animation_frame='Month_Name',
    animation_group='City',
    range_y=[3, 8],
    title='Monthly Solar Irradiance by City — West Africa (Animated by Month)',
    labels={
        'GHI': 'Mean GHI (kWh/m²/day)',
        'City': 'City',
        'Climate_Zone': 'Climate Zone',
        'Month_Name': 'Month'
    },
    category_orders={'City': city_summary['City'].tolist()}
)

fig3.update_layout(
    height=550,
    plot_bgcolor='white',
    paper_bgcolor='white',
    font=dict(family='Arial', size=11),
    title_font_size=14,
    xaxis=dict(tickangle=45),
    showlegend=True
)

show_chart(fig3, 'plotly_animated_monthly.html')

✓ Chart saved and opened: plotly_animated_monthly.html


In [7]:
import subprocess
subprocess.run(['pip', 'install', 'statsmodels'])

CompletedProcess(args=['pip', 'install', 'statsmodels'], returncode=0)

In [8]:
# ================================================
# CHART 4 — Scatter Plot: Solar vs Electricity Access
# ================================================

# Load electricity access data
wb_df = pd.read_csv('../data/API_EG.ELC.ACCS.ZS_DS2_en_csv_v2_127016.csv',
                    skiprows=4)

country_name_map = {
    'Nigeria':       'Nigeria',
    'Ghana':         'Ghana',
    'Senegal':       'Senegal',
    "Cote d'Ivoire": "Cote d'Ivoire",
    'Mali':          'Mali',
    'Burkina Faso':  'Burkina Faso',
    'Niger':         'Niger',
    'Guinea':        'Guinea',
    'Togo':          'Togo',
    'Benin':         'Benin',
    'Sierra Leone':  'Sierra Leone',
    'Liberia':       'Liberia',
    'Mauritania':    'Mauritania',
    'Gambia':        'Gambia, The',
    'Guinea-Bissau': 'Guinea-Bissau',
    'Cape Verde':    'Cabo Verde',
}

wa_access = wb_df[wb_df['Country Name'].isin(country_name_map.values())]
wa_access = wa_access[['Country Name', '2023']].dropna()
wa_access.columns = ['WB_Name', 'Electricity_Access_2023']
display_map = {v: k for k, v in country_name_map.items()}
wa_access['Country'] = wa_access['WB_Name'].map(display_map)

# Merge with solar data
merged = city_summary.merge(wa_access[['Country', 'Electricity_Access_2023']],
                             on='Country', how='left')

fig4 = px.scatter(
    merged,
    x='Electricity_Access_2023',
    y='Mean_GHI',
    color='Climate_Zone',
    color_discrete_map=zone_colors,
    size='Mean_GHI',
    hover_name='City',
    hover_data={
        'Country': True,
        'Mean_GHI': ':.3f',
        'Electricity_Access_2023': ':.1f',
        'Climate_Zone': True,
    },
    title='Solar Potential vs Electricity Access — 18 West African Cities (2023)',
    labels={
        'Electricity_Access_2023': 'Electricity Access (% of population, 2023)',
        'Mean_GHI': 'Annual Average GHI (kWh/m²/day)',
        'Climate_Zone': 'Climate Zone'
    },
    trendline='ols'
)

fig4.update_layout(
    height=550,
    plot_bgcolor='white',
    paper_bgcolor='white',
    font=dict(family='Arial', size=12),
    title_font_size=14,
    xaxis=dict(showgrid=True, gridcolor='#eeeeee'),
    yaxis=dict(showgrid=True, gridcolor='#eeeeee'),
)

show_chart(fig4, 'plotly_solar_vs_access.html')

✓ Chart saved and opened: plotly_solar_vs_access.html


In [9]:
# ================================================
# CHART 5 — Plotly Dash
# ================================================

import subprocess
subprocess.run(['pip', 'install', 'dash'])

CompletedProcess(args=['pip', 'install', 'dash'], returncode=0)

In [10]:
# ================================================
# DASH APP — Simple Solar Dashboard
# ================================================

from dash import Dash, dcc, html, Input, Output
import plotly.express as px
import pandas as pd

# Load data
df = pd.read_csv('../data/west_africa_solar_data_2014_2025.csv')
city_summary = df.groupby(['City', 'Country', 
                            'Climate_Zone'])['GHI'].mean().reset_index()
city_summary.columns = ['City', 'Country', 'Climate_Zone', 'Mean_GHI']

zone_colors = {
    'Coastal Humid': '#0099ff',
    'Savanna':       '#00c27c',
    'Sahel':         '#f7c948',
    'Desert/Arid':   '#ff6b35'
}

# Initialise app
app = Dash(__name__)

app.layout = html.Div([

    # Title
    html.H1("West Africa Solar Potential Dashboard",
            style={'textAlign': 'center', 'fontFamily': 'Arial',
                   'color': '#04080f', 'padding': '20px'}),

    # Dropdown filter
    html.Div([
        html.Label("Filter by Climate Zone:",
                   style={'fontFamily': 'Arial', 'fontWeight': 'bold'}),
        dcc.Dropdown(
            id='zone-filter',
            options=[{'label': z, 'value': z}
                     for z in city_summary['Climate_Zone'].unique()],
            value=None,
            placeholder="Select a climate zone...",
            clearable=True,
            style={'width': '400px'}
        )
    ], style={'padding': '20px'}),

    # Bar chart
    dcc.Graph(id='bar-chart'),

    # Summary text
    html.Div(id='summary-text',
             style={'padding': '20px', 'fontFamily': 'Arial',
                    'fontSize': '14px', 'color': '#444'})

], style={'backgroundColor': '#f9f9f9', 'minHeight': '100vh'})


# Callback — update chart based on dropdown
@app.callback(
    Output('bar-chart', 'figure'),
    Output('summary-text', 'children'),
    Input('zone-filter', 'value')
)
def update_chart(selected_zone):
    if selected_zone:
        filtered = city_summary[city_summary['Climate_Zone'] == selected_zone]
    else:
        filtered = city_summary

    filtered = filtered.sort_values('Mean_GHI', ascending=True)

    fig = px.bar(
        filtered,
        x='Mean_GHI',
        y='City',
        color='Climate_Zone',
        color_discrete_map=zone_colors,
        orientation='h',
        text='Mean_GHI',
        labels={'Mean_GHI': 'Annual Avg GHI (kWh/m²/day)', 'City': ''}
    )
    fig.update_traces(texttemplate='%{text:.2f}', textposition='outside')
    fig.update_layout(
        plot_bgcolor='white',
        paper_bgcolor='white',
        height=500,
        showlegend=False
    )

    avg = filtered['Mean_GHI'].mean()
    summary = f"Showing {len(filtered)} cities — Average GHI: {avg:.2f} kWh/m²/day"

    return fig, summary


# Run app
if __name__ == '__main__':
    app.run(debug=True, port=8050)

print("✓ Dash app ready — open http://127.0.0.1:8050 in your browser")

✓ Dash app ready — open http://127.0.0.1:8050 in your browser


## Plotly & Dash: Key Learnings

### Plotly express vs Matplotlib
Matplotlib gives a complete control but requires many lines of code for every element. Plotly Express produces interactive charts in a fraction of the code (hover tooltips, zoom, pan, and download are built in automatically). The tradeoff is less fine-grained control over styling.

### The Animated chart
The animation_frame parameter in Plotly Express is one of the most powerful data storytelling tools available. Watching solar irradiance shift month by month across 18 cities makes seasonal patterns immediately intuitive in a way static charts cannot. The January-to-July transition tells the West African monsoon story better than any written description.

### Plotly dash: From chart to application
Dash takes Plotly charts and wraps them in a web application framework. The callback system, where user inputs trigger chart updates is the foundation of professional data applications.

In [11]:
# ================================================
# ARTICLE CHART 1 — Seasonal Solar Variation
# Monthly GHI trend for Lagos, Abuja, Kano
# ================================================

nigeria_cities = ['Lagos', 'Abuja', 'Kano']
nigeria_colors = {
    'Lagos': '#0099ff',
    'Abuja': '#00c27c', 
    'Kano':  '#f7c948'
}

month_names = ['Jan','Feb','Mar','Apr','May','Jun',
               'Jul','Aug','Sep','Oct','Nov','Dec']

# Monthly averages for Nigerian cities only
nigeria_monthly = df[df['City'].isin(nigeria_cities)].groupby(
    ['City', 'Month'])['GHI'].mean().reset_index()
nigeria_monthly['Month_Name'] = nigeria_monthly['Month'].map(
    dict(enumerate(month_names, 1)))
nigeria_monthly['Month_Name'] = pd.Categorical(
    nigeria_monthly['Month_Name'],
    categories=month_names, ordered=True)
nigeria_monthly = nigeria_monthly.sort_values('Month_Name')

fig_c1 = px.line(
    nigeria_monthly,
    x='Month_Name',
    y='GHI',
    color='City',
    color_discrete_map=nigeria_colors,
    markers=True,
    title='Nigeria\'s Solar Divide — Monthly Irradiance by City (2014–2025)',
    labels={
        'GHI': 'Mean GHI (kWh/m²/day)',
        'Month_Name': 'Month',
        'City': 'City'
    }
)

# Add shaded rainy season
fig_c1.add_vrect(
    x0='Jun', x1='Sep',
    fillcolor='lightblue', opacity=0.15,
    layer='below', line_width=0,
    annotation_text='Rainy Season',
    annotation_position='top left',
    annotation_font_size=11,
    annotation_font_color='#0099ff'
)

# Add annotations for peak months
for city in nigeria_cities:
    city_data = nigeria_monthly[nigeria_monthly['City'] == city]
    peak_row = city_data.loc[city_data['GHI'].idxmax()]
    fig_c1.add_annotation(
        x=peak_row['Month_Name'],
        y=peak_row['GHI'],
        text=f"Peak: {peak_row['GHI']:.2f}",
        showarrow=True,
        arrowhead=2,
        arrowcolor=nigeria_colors[city],
        font=dict(size=10, color=nigeria_colors[city]),
        bgcolor='white',
        bordercolor=nigeria_colors[city],
        borderwidth=1,
        ay=-30
    )

fig_c1.update_layout(
    height=500,
    plot_bgcolor='white',
    paper_bgcolor='white',
    font=dict(family='Arial', size=12),
    title_font_size=15,
    title_font_family='Arial',
    xaxis=dict(showgrid=True, gridcolor='#f0f0f0'),
    yaxis=dict(showgrid=True, gridcolor='#f0f0f0',
               range=[3.0, 7.5]),
    legend=dict(
        orientation='h',
        yanchor='bottom',
        y=1.02,
        xanchor='right',
        x=1
    ),
    hovermode='x unified'
)

show_chart(fig_c1, 'article_chart1_seasonal.html')

✓ Chart saved and opened: article_chart1_seasonal.html


In [12]:
# ================================================
# ARTICLE CHART 2 — Nigeria Solar Gradient Map
# ================================================

# Nigerian cities with GHI values
nigeria_data = city_summary[city_summary['City'].isin(nigeria_cities)].copy()
nigeria_coords = {
    'Lagos': (6.5244,  3.3792),
    'Abuja': (9.0765,  7.3986),
    'Kano':  (12.0022, 8.5920),
}
nigeria_data['lat'] = nigeria_data['City'].map(
    lambda c: nigeria_coords[c][0])
nigeria_data['lon'] = nigeria_data['City'].map(
    lambda c: nigeria_coords[c][1])

# Add descriptive labels
nigeria_data['Label'] = nigeria_data.apply(
    lambda r: f"{r['City']}<br>{r['Mean_GHI']:.2f} kWh/m²/day<br>{r['Climate_Zone']}",
    axis=1
)

fig_c2 = go.Figure()

# Add scatter mapbox points
fig_c2 = px.scatter_mapbox(
    nigeria_data,
    lat='lat',
    lon='lon',
    size='Mean_GHI',
    color='Mean_GHI',
    color_continuous_scale='YlOrRd',
    hover_name='City',
    hover_data={
        'Mean_GHI': ':.3f',
        'Climate_Zone': True,
        'lat': False,
        'lon': False
    },
    zoom=5,
    center={'lat': 9.5, 'lon': 7.5},
    title='Nigeria Solar Irradiance Gradient — North Outperforms South by 30%',
    labels={'Mean_GHI': 'Annual Avg GHI (kWh/m²/day)'},
    size_max=60,
    mapbox_style='carto-positron'
)

# Add city labels
for _, row in nigeria_data.iterrows():
    fig_c2.add_trace(go.Scattermapbox(
        lat=[row['lat']],
        lon=[row['lon']],
        mode='text',
        text=[f"  {row['City']}<br>  {row['Mean_GHI']:.2f} kWh/m²/day"],
        textfont=dict(size=13, color='#333333'),
        showlegend=False,
        hoverinfo='skip'
    ))

fig_c2.update_layout(
    height=550,
    font=dict(family='Arial', size=12),
    title_font_size=15,
    coloraxis_colorbar=dict(
        title='GHI<br>(kWh/m²/day)',
        thickness=15
    ),
    margin=dict(l=0, r=0, t=50, b=0)
)

show_chart(fig_c2, 'article_chart2_nigeria_map.html')

C:\Users\User\AppData\Local\Temp\ipykernel_12932\1791512352.py:26: DeprecationWarning: *scatter_mapbox* is deprecated! Use *scatter_map* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/
  fig_c2 = px.scatter_mapbox(
C:\Users\User\AppData\Local\Temp\ipykernel_12932\1791512352.py:50: DeprecationWarning: *scattermapbox* is deprecated! Use *scattermap* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/
  fig_c2.add_trace(go.Scattermapbox(


✓ Chart saved and opened: article_chart2_nigeria_map.html


In [13]:
# ================================================
# ARTICLE CHART 3 — Solar Potential vs Access
# West Africa scatter plot
# ================================================

# Add Nigeria labels
merged['Label'] = merged['City'].apply(
    lambda x: f'⭐ {x}' if x in nigeria_cities else x
)

merged['Is_Nigeria'] = merged['Country'] == 'Nigeria'

fig_c3 = px.scatter(
    merged,
    x='Electricity_Access_2023',
    y='Mean_GHI',
    color='Climate_Zone',
    color_discrete_map=zone_colors,
    size='Mean_GHI',
    size_max=25,
    hover_name='City',
    hover_data={
        'Country': True,
        'Mean_GHI': ':.3f',
        'Electricity_Access_2023': ':.1f',
        'Climate_Zone': True,
    },
    title='High Solar Potential + Low Electricity Access = The Opportunity Gap',
    labels={
        'Electricity_Access_2023': 'Electricity Access (% population, 2023)',
        'Mean_GHI': 'Annual Average GHI (kWh/m²/day)',
        'Climate_Zone': 'Climate Zone'
    },
    trendline='ols',
    trendline_scope='overall',
    trendline_color_override='#999999'
)

# Annotate Nigerian cities
for _, row in merged[merged['Country'] == 'Nigeria'].iterrows():
    fig_c3.add_annotation(
        x=row['Electricity_Access_2023'],
        y=row['Mean_GHI'],
        text=f"  {row['City']}",
        showarrow=False,
        font=dict(size=11, color='#333333', family='Arial'),
        xanchor='left'
    )

# Add quadrant labels
fig_c3.add_annotation(
    x=25, y=6.1,
    text="High Solar<br>Low Access<br>= Opportunity",
    showarrow=False,
    font=dict(size=11, color='#ff6b35'),
    bgcolor='rgba(255,107,53,0.1)',
    bordercolor='#ff6b35',
    borderwidth=1,
    borderpad=6
)

fig_c3.add_annotation(
    x=80, y=4.7,
    text="High Access<br>Lower Solar",
    showarrow=False,
    font=dict(size=11, color='#0099ff'),
    bgcolor='rgba(0,153,255,0.1)',
    bordercolor='#0099ff',
    borderwidth=1,
    borderpad=6
)

# Add average lines
avg_access = merged['Electricity_Access_2023'].mean()
avg_ghi = merged['Mean_GHI'].mean()

fig_c3.add_hline(
    y=avg_ghi, line_dash='dash',
    line_color='gray', line_width=1,
    annotation_text=f'Avg GHI: {avg_ghi:.2f}',
    annotation_position='right'
)

fig_c3.add_vline(
    x=avg_access, line_dash='dash',
    line_color='gray', line_width=1,
    annotation_text=f'Avg Access: {avg_access:.1f}%',
    annotation_position='top'
)

fig_c3.update_layout(
    height=580,
    plot_bgcolor='white',
    paper_bgcolor='white',
    font=dict(family='Arial', size=12),
    title_font_size=15,
    xaxis=dict(showgrid=True, gridcolor='#f0f0f0'),
    yaxis=dict(showgrid=True, gridcolor='#f0f0f0'),
    legend=dict(
        orientation='h',
        yanchor='bottom',
        y=1.02,
        xanchor='right',
        x=1
    )
)

show_chart(fig_c3, 'article_chart3_solar_vs_access.html')

✓ Chart saved and opened: article_chart3_solar_vs_access.html


In [14]:
import subprocess
subprocess.run(['pip', 'install', 'kaleido'])

CompletedProcess(args=['pip', 'install', 'kaleido'], returncode=0)

In [ ]:
# ================================================
# EXPORT CHARTS AS PNG 
# ================================================

# Chart 1 — Seasonal variation
fig_c1.write_image('../reports/article_chart1_seasonal.png', 
                   width=1200, height=500, scale=2)
print("✓ Chart 1 saved")

# Chart 2 — Nigeria map
fig_c2.write_image('../reports/article_chart2_nigeria_map.png',
                   width=1200, height=550, scale=2)
print("✓ Chart 2 saved")

# Chart 3 — Scatter plot
fig_c3.write_image('../reports/article_chart3_solar_vs_access.png',
                   width=1200, height=580, scale=2)
print("✓ Chart 3 saved")

print("\n✓ All charts exported at 2400x1000px — publication quality")

C:\Users\User\AppData\Local\Temp\ipykernel_12932\1944503183.py:6: DeprecationWarning: 
Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).

  fig_c1.write_image('../reports/article_chart1_seasonal.png',


In [ ]:
# Export Chart 1 only
fig_c1.write_image('../reports/article_chart1_seasonal.png',
                   width=1200, height=500, scale=2)
print("✓ Chart 1 saved")

C:\Users\User\AppData\Local\Temp\ipykernel_15012\3996896269.py:2: DeprecationWarning: 
Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).

  fig_c1.write_image('../reports/article_chart1_seasonal.png',
